In [1]:
import os
import ast
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
#Setting Visual Styles for the charts
sns.set_theme(style='darkgrid')
plt.rcParams.update({'font.sans-serif':'DejaVu Sans','font.family':'sans-serif'})

### 1. Loading Cleaned Datasets

In [6]:
df_users=pd.read_csv('../data/cleaned_data_/users_clean.csv')
df_bookings=pd.read_csv('../data/cleaned_data_/bookings_clean.csv')
df_snacks=pd.read_csv('../data/cleaned_data_/snacks_clean.csv')
df_reviews=pd.read_csv('../data/cleaned_data_/reviews_clean.csv')
df_chats=pd.read_csv('../data/cleaned_data_/chats_clean.csv')

print(f"✅ Loaded {len(df_users)} users, {len(df_bookings)} bookings, {len(df_snacks)} snack orders, {len(df_reviews)} reviews, and {len(df_chats)} chat logs.")

✅ Loaded 45 users, 215 bookings, 150 snack orders, 42 reviews, and 65 chat logs.


### Chart 1. Revenue Contribution Per User

In [8]:
b_spend = df_bookings.groupby('user_id')['total'].sum().reset_index(name='booking_spend')
s_spend = df_snacks.groupby('user_id')['total'].sum().reset_index(name='snack_spend')
user_rev = df_users[['id', 'username']].merge(b_spend, left_on='id', right_on='user_id', how='left')
user_rev = user_rev.merge(s_spend, left_on='id', right_on='user_id', how='left').fillna(0)
user_rev['total_spend'] = user_rev['booking_spend'] + user_rev['snack_spend']
user_rev = user_rev.sort_values('total_spend', ascending=False)
plt.figure(figsize=(14, 6))
barplot = sns.barplot(data=user_rev.head(20), x='username', y='total_spend', palette='magma')
plt.xticks(rotation=45, ha='right', fontsize=10)
plt.title('Top 20 Customers by Lifetime Spend (Tickets + Concessions)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Username', fontsize=11, fontweight='bold')
plt.ylabel('Total Spend ($)', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('C:/CLV-Project/data/charts/01_top_users_revenue.png', dpi=300)
plt.close()

C:\Users\Admin\AppData\Local\Temp\ipykernel_51608\3460821811.py:8: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  barplot = sns.barplot(data=user_rev.head(20), x='username', y='total_spend', palette='magma')


In [9]:
# Print Pareto principle stat
top_20_pct_count = int(np.ceil(len(user_rev) * 0.2))
top_20_pct_rev = user_rev.head(top_20_pct_count)['total_spend'].sum()
total_rev = user_rev['total_spend'].sum()
print(f"   💡 Insight: Top 20% of users ({top_20_pct_count} users) generate {(top_20_pct_rev/total_rev)*100:.1f}% of total cinema revenue!")

   💡 Insight: Top 20% of users (9 users) generate 48.8% of total cinema revenue!


### Showtime & Day HeatMap

In [14]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Make sure created_at is datetime
df_bookings['created_at'] = pd.to_datetime(df_bookings['created_at'])

# Extract day of week
df_bookings['booking_day'] = df_bookings['created_at'].dt.day_name()

# Correct day order
day_order = [
    'Monday',
    'Tuesday',
    'Wednesday',
    'Thursday',
    'Friday',
    'Saturday',
    'Sunday'
]

# Create pivot table
heatmap_data = df_bookings.pivot_table(
    index='booking_day',
    columns='showtime',
    values='tickets',
    aggfunc='sum',
    fill_value=0
).reindex(day_order)

# Plot
plt.figure(figsize=(10, 6))

sns.heatmap(
    heatmap_data,
    annot=True,
    fmt='.0f',
    cmap='YlOrRd',
    cbar_kws={'label': 'Tickets Sold'},
    linewidths=0.5
)

plt.title(
    'Ticket Sales Heatmap (Day of Week vs Showtime)',
    fontsize=14,
    fontweight='bold',
    pad=15
)

plt.xlabel('Showtime Slot', fontsize=11, fontweight='bold')
plt.ylabel('Day of Week', fontsize=11, fontweight='bold')

plt.tight_layout()


# Save
plt.savefig(
    'C:/CLV-Project/data/charts/02_showtime_heatmap.png',
    dpi=300,
    bbox_inches='tight'
)

plt.close()

### Chart 3: Theatre Revenue BreakDown

In [16]:
theater_stats = df_bookings.groupby('theater').agg(
    total_revenue=('total', 'sum'),
    total_tickets=('tickets', 'sum'),
    booking_count=('id', 'count')
).sort_values('total_revenue', ascending=False).reset_index()
fig, ax1 = plt.subplots(figsize=(10, 5))
color = 'tab:blue'
ax1.set_xlabel('Theater Screen Name', fontweight='bold', fontsize=11)
ax1.set_ylabel('Total Revenue ($)', color=color, fontweight='bold', fontsize=11)
bars = ax1.bar(theater_stats['theater'], theater_stats['total_revenue'], color=color, alpha=0.8, width=0.4, align='center')
ax1.tick_params(axis='y', labelcolor=color)
ax2 = ax1.twinx()  
color = 'tab:red'
ax2.set_ylabel('Tickets Sold', color=color, fontweight='bold', fontsize=11)
lines = ax2.plot(theater_stats['theater'], theater_stats['total_tickets'], color=color, marker='o', linewidth=2.5, markersize=8)
ax2.tick_params(axis='y', labelcolor=color)
plt.title('Revenue & Ticket Volume by Theater Screen', fontsize=14, fontweight='bold', pad=15)
fig.tight_layout()
plt.savefig('C:/CLV-Project/data/charts/03_theater_performance.png', dpi=300)
plt.close()

### Chart 4: Snack Item Popularity

In [17]:
all_items = []
for row in df_snacks['items']:
    try:
        parsed = ast.literal_eval(row) if isinstance(row, str) and row.startswith('[') else json.loads(row) if isinstance(row, str) else row
        if isinstance(parsed, list):
            all_items.extend(parsed)
        else:
            all_items.append(str(parsed))
    except:
        all_items.append(str(row))
item_counts = pd.Series(all_items).value_counts().reset_index()
item_counts.columns = ['item_name', 'order_count']
plt.figure(figsize=(10, 5))
sns.barplot(data=item_counts, x='order_count', y='item_name', palette='viridis')
plt.title('Most Popular Concession Snacks & Beverages', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Number of Orders', fontsize=11, fontweight='bold')
plt.ylabel('Snack Item', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('C:/CLV-Project/data/charts/04_snack_popularity.png', dpi=300)
plt.close()

C:\Users\Admin\AppData\Local\Temp\ipykernel_51608\1840825990.py:14: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=item_counts, x='order_count', y='item_name', palette='viridis')


### Chart 5: Review Rating Distribution

In [18]:
plt.figure(figsize=(8, 5))
rating_counts = df_reviews['rating'].value_counts().sort_index()
ax = sns.barplot(x=rating_counts.index, y=rating_counts.values, palette='YlGn_r')
plt.title('Customer Review Ratings Breakdown (1 to 5 Stars)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Star Rating', fontsize=11, fontweight='bold')
plt.ylabel('Number of Reviews', fontsize=11, fontweight='bold')
# Annotate count numbers above bars
for p in ax.patches:
    ax.annotate(f'{int(p.get_height())}', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='center', xytext=(0, 5), textcoords='offset points', fontweight='bold')
plt.tight_layout()
plt.savefig('C:/CLV-Project/data/charts/05_review_ratings.png', dpi=300)
plt.close()

C:\Users\Admin\AppData\Local\Temp\ipykernel_51608\3026786486.py:3: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  ax = sns.barplot(x=rating_counts.index, y=rating_counts.values, palette='YlGn_r')


### Chart 6: Monthly Revenue Timeline

In [22]:
df_bookings['created_at'] = pd.to_datetime(df_bookings['created_at'])
df_snacks['created_at'] = pd.to_datetime(df_snacks['created_at'])

df_bookings['booking_month'] = df_bookings['created_at'].dt.to_period('M')
df_snacks['order_month'] = df_snacks['created_at'].dt.to_period('M')

b_monthly = (
    df_bookings.groupby('booking_month')['total']
    .sum()
    .reset_index(name='booking_rev')
)

s_monthly = (
    df_snacks.groupby('order_month')['total']
    .sum()
    .reset_index(name='snack_rev')
)

# Rename both to common column name
b_monthly = b_monthly.rename(columns={'booking_month': 'month_year'})
s_monthly = s_monthly.rename(columns={'order_month': 'month_year'})

monthly_trend = (
    b_monthly
    .merge(s_monthly, on='month_year', how='outer')
    .fillna(0)
)

monthly_trend['month_str'] = monthly_trend['month_year'].astype(str)

monthly_trend['total_rev'] = (
    monthly_trend['booking_rev'] +
    monthly_trend['snack_rev']
)

# Sort chronologically
monthly_trend = monthly_trend.sort_values('month_year')

plt.figure(figsize=(11, 5))

plt.plot(
    monthly_trend['month_str'],
    monthly_trend['booking_rev'],
    marker='o',
    linewidth=2,
    label='Ticket Revenue',
    color='dodgerblue'
)

plt.plot(
    monthly_trend['month_str'],
    monthly_trend['snack_rev'],
    marker='s',
    linewidth=2,
    label='Snack Revenue',
    color='orange'
)

plt.plot(
    monthly_trend['month_str'],
    monthly_trend['total_rev'],
    marker='^',
    linewidth=3,
    linestyle='--',
    label='Total Revenue',
    color='green'
)

plt.title(
    'Monthly Revenue Timeline (Jan 2026 – July 2026)',
    fontsize=14,
    fontweight='bold',
    pad=15
)

plt.xlabel('Month', fontsize=11, fontweight='bold')
plt.ylabel('Revenue ($)', fontsize=11, fontweight='bold')

plt.legend(fontsize=11)
plt.tight_layout()

plt.savefig(
    'C:/CLV-Project/data/charts/06_monthly_revenue_trend.png',
    dpi=300
)

plt.close()